# 10 — Filesystem-cache startup

Filesystem page-cache state is distinct from compiled-component cache state. No persistent compiled-cache claim is made without artifact identity and hit evidence. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

try: batch=resolve_result_batch('e-perf-9', diagnostic_path=os.environ.get('E_PERF_9_DIR'))
except (FileNotFoundError, RuntimeError, ValueError): batch=None
by_condition={}
if batch is not None:
    for path,value in passed_json(batch,'startup.json'): by_condition[path.parent.parent.name]=value
rows=[]
for condition in ['small-cold','small-warm','medium-cold','medium-warm','large-cold','large-warm']:
    value=by_condition.get(condition)
    if value is None:
        row=pending_record(condition,'no passed startup.json leaf','milliseconds'); row['condition']=condition; rows.append(row)
    else:
        row={'question':condition,'condition':condition,'status':'READY','cache_state':value['cache_state'],'total_ms':value['total_wall_duration_ns']/1e6,'compiled_cache_mode':value['compiled_component_cache']['mode'],'compiled_cache_hit':value['compiled_component_cache']['hit'],'units':'milliseconds','uncertainty':'descriptive only','thesis_evidence':False}
        row.update({f'{name}_ms':duration/1e6 for name,duration in value['phases_ns'].items()}); rows.append(row)
df=pd.DataFrame(rows); print(evidence_label(len(df[df.status=='READY']), 'milliseconds', False)); display(df)
